# 🐘 Elephant Rumble Denoiser - Interactive Notebook

This notebook provides an interactive interface for testing and visualizing the denoising pipeline.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

from config.config import CONFIG
from src.batch_process import load_and_validate_data
from src.pipeline import process_single_call
from src.tuning import test_parameters

from IPython.display import Image, Audio, display
import pandas as pd

## 1. Configuration

In [ ]:
# Display current configuration
CONFIG.print_summary()

## 2. Load Data

In [ ]:
# Update these paths
CSV_PATH = '/path/to/your/calls.csv'
AUDIO_FOLDER = '/path/to/your/audio/folder'

df = load_and_validate_data(CSV_PATH, AUDIO_FOLDER)
df.head()

## 3. Test Single Call

In [ ]:
# Pick first call
test_row = df.iloc[0]

result = process_single_call(
    audio_path=test_row['file_path'],
    start_time=test_row['Start_time'],
    end_time=test_row['End_time'],
    selection_id=test_row['Selection']
)

# Display result
for key, value in result.items():
    if key != 'noise_validation':
        print(f'{key:20s}: {value}')

## 4. Visualize Results

In [ ]:
if result['status'] == 'success':
    # Display comparison spectrogram
    print('Before/After Comparison:')
    display(Image(result['comparison_plot']))
    
    # Play cleaned audio
    print('\nCleaned Audio:')
    display(Audio(result['output_audio']))

## 5. Parameter Tuning

In [ ]:
# Test different parameter combinations
tuning_results = test_parameters(
    df,
    selection_id=1,
    prop_decrease_values=[0.75, 0.85, 0.90],
    hpss_margins=[2.0, 3.0, 4.0]
)

tuning_results[['prop_decrease', 'hpss_margin', 'status', 'duration']]

## 6. Batch Processing

In [ ]:
from src.batch_process import batch_process, print_analysis

# Process all calls
results_df = batch_process(df, output_dir='../outputs')

# Print analysis
print_analysis(results_df)